# Benchmark — Naive Bayes MapReduce (All Four Implementations)

This notebook times all four implementations across three dataset sizes
and prints a comparison table.

**Experiment design** (following Zheng 2014):
- Sizes are created by repeating the base dataset: ×1, ×10, ×50
- This simulates small / medium / large workloads with the same class distribution
- All implementations use the same train/test split (fixed seed)

**What to look for in the results:**
- Does the optimised RDD outperform the baseline consistently at all scales?
- At what scale (if any) does the DataFrame API overtake the RDD API?
- Does prediction time scale linearly with dataset size, or does it stay flat
  (prediction does not use MapReduce, only broadcast lookups)?

In [ ]:
# Cell 1 — Imports and SparkSession setup
import time
import sys
import os

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

sys.path.insert(0, os.path.abspath('..'))

from core.naive_bayes import compute_log_probs, predict, evaluate
from data.loader import load_rdd, load_dataframe, FEATURE_COLS, LABEL_COL

# TODO (Databricks): Remove .master() and paste your cluster config here.
# Example cluster config:
#   spark = SparkSession.builder \
#       .appName('NaiveBayes-Benchmark') \
#       .config('spark.executor.cores', '4') \
#       .config('spark.executor.memory', '8g') \
#       .getOrCreate()
spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('NaiveBayes-Benchmark')
    .config('spark.sql.shuffle.partitions', '8')
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel('WARN')
print('SparkSession ready.')

In [ ]:
# Cell 2 — Experiment configuration
#
# TODO: Replace DATA_PATH with your actual dataset file.
#   Local      : '/Users/you/data/car.data'
#   Databricks : 'dbfs:/FileStore/car.data'
# Leave as None to run the 5-row dummy dataset (smoke test only).
DATA_PATH = None

# TODO: Adjust SCALE_FACTORS for your environment.
#   With dummy data (5 rows)    : [1, 2, 3]  — just verifies the code runs
#   With real data (1728 rows)  : [1, 10, 50]  — gives ~1.7k, ~17k, ~86k rows
#   On Databricks with large data: [1, 100, 500]  — for meaningful scalability results
SCALE_FACTORS = [1, 2, 3]  # TODO: change to [1, 10, 50] for real experiments

print(f'Data path     : {DATA_PATH or "dummy (5 rows)"}')
print(f'Scale factors : {SCALE_FACTORS}')

In [ ]:
# Cell 3 — Training and prediction helpers (inline, one per implementation)
#
# Each function reproduces the core logic from the corresponding notebook.
# Keeping them here makes the benchmark self-contained without requiring
# notebook imports (which PySpark does not support natively).

# ── RDD Baseline ──────────────────────────────────────────────────────────
# flatMap + groupByKey: unoptimised shuffle, no broadcast, no persist.

def train_rdd_baseline(train_rdd):
    """MapReduce training using flatMap + groupByKey (baseline)."""
    def map_row(row):
        label, features = row
        pairs = []
        for feat_idx, value in enumerate(features):
            pairs.append((f'feat_{feat_idx}_{value}_{label}', 1))
        pairs.append((f'class_{label}', 1))
        pairs.append(('total', 1))
        return pairs

    mapped  = train_rdd.flatMap(map_row)
    grouped = mapped.groupByKey()
    counts  = grouped.mapValues(sum)
    return dict(counts.collect())


def predict_rdd_baseline(test_rdd, log_prob_table):
    """Prediction without broadcast — log_prob_table is serialised per task."""
    pairs = test_rdd.map(lambda row: (row[0], predict(log_prob_table, row[1]))).collect()
    return pairs


# ── RDD Optimised ─────────────────────────────────────────────────────────
# mapPartitions + reduceByKey + broadcast + persist.

def train_rdd_optimized(train_rdd):
    """MapReduce training using mapPartitions + reduceByKey (optimised)."""
    def map_partition(rows):
        local_counts = {}
        for label, features in rows:
            for feat_idx, value in enumerate(features):
                key = f'feat_{feat_idx}_{value}_{label}'
                local_counts[key] = local_counts.get(key, 0) + 1
            class_key = f'class_{label}'
            local_counts[class_key] = local_counts.get(class_key, 0) + 1
            local_counts['total']   = local_counts.get('total', 0) + 1
        yield from local_counts.items()

    mapped    = train_rdd.mapPartitions(map_partition)
    counts    = mapped.reduceByKey(lambda a, b: a + b)
    counts.persist()
    result    = dict(counts.collect())
    counts.unpersist()
    return result


def predict_rdd_optimized(test_rdd, log_prob_table):
    """Prediction with broadcast — log_prob_table sent once per executor."""
    bc = sc.broadcast(log_prob_table)
    pairs = test_rdd.map(lambda row: (row[0], predict(bc.value, row[1]))).collect()
    bc.unpersist()
    return pairs


# ── DataFrame Baseline ────────────────────────────────────────────────────
# UDF key builder + groupBy: opaque to Catalyst, no persist, no repartition.

def train_df_baseline(train_df):
    """DataFrame training using a Python UDF to build key strings (baseline)."""
    make_key_udf = F.udf(
        lambda feat_idx, value, label: f'feat_{feat_idx}_{value}_{label}',
        StringType()
    )
    class_counts_df = train_df.groupBy(LABEL_COL).agg(F.count('*').alias('class_count'))
    key_dfs = [
        train_df.select(
            make_key_udf(F.lit(i), F.col(c), F.col(LABEL_COL)).alias('key')
        )
        for i, c in enumerate(FEATURE_COLS)
    ]
    all_keys_df = key_dfs[0]
    for df in key_dfs[1:]:
        all_keys_df = all_keys_df.union(df)
    feature_counts_df = all_keys_df.groupBy('key').agg(F.count('*').alias('count'))

    class_counts   = {r[LABEL_COL]: r['class_count'] for r in class_counts_df.collect()}
    feature_counts = {r['key']: r['count'] for r in feature_counts_df.collect()}
    return class_counts, feature_counts


def predict_df_baseline(test_df, log_prob_table):
    """Prediction using a Python UDF without broadcast."""
    pred_udf = F.udf(
        lambda *features: predict(log_prob_table, list(features)),
        StringType()
    )
    result_df = test_df.withColumn('prediction', pred_udf(*[F.col(c) for c in FEATURE_COLS]))
    return result_df.select(LABEL_COL, 'prediction').collect()


# ── DataFrame Optimised ───────────────────────────────────────────────────
# F.concat key builder + persist + broadcast.

def train_df_optimized(train_df):
    """DataFrame training using F.concat (Catalyst-friendly) + persist (optimised)."""
    class_counts_df = train_df.groupBy(LABEL_COL).agg(F.count('*').alias('class_count'))
    key_dfs = [
        train_df.select(
            F.concat(F.lit(f'feat_{i}_'), F.col(c), F.lit('_'), F.col(LABEL_COL)).alias('key')
        )
        for i, c in enumerate(FEATURE_COLS)
    ]
    all_keys_df = key_dfs[0]
    for df in key_dfs[1:]:
        all_keys_df = all_keys_df.union(df)
    feature_counts_df = all_keys_df.groupBy('key').agg(F.count('*').alias('count'))
    feature_counts_df.persist()

    class_counts   = {r[LABEL_COL]: r['class_count'] for r in class_counts_df.collect()}
    feature_counts = {r['key']: r['count'] for r in feature_counts_df.collect()}
    feature_counts_df.unpersist()
    return class_counts, feature_counts


def predict_df_optimized(test_df, log_prob_table):
    """Prediction using a Python UDF with broadcast."""
    bc = sc.broadcast(log_prob_table)
    pred_udf = F.udf(
        lambda *features: predict(bc.value, list(features)),
        StringType()
    )
    result_df = test_df.withColumn('prediction', pred_udf(*[F.col(c) for c in FEATURE_COLS]))
    rows = result_df.select(LABEL_COL, 'prediction').collect()
    bc.unpersist()
    return rows


print('Helper functions defined.')

In [ ]:
# Cell 4 — Run all experiments
#
# For each scale factor we:
#   1. Load the base dataset
#   2. Repeat it scale_factor times to create a larger dataset
#   3. Time training + prediction for all four implementations
#   4. Record results for the summary table

SIZE_LABELS = {SCALE_FACTORS[0]: 'Small', SCALE_FACTORS[1]: 'Medium', SCALE_FACTORS[2]: 'Large'}
results = []  # list of dicts, one per (implementation, scale) combination

for scale in SCALE_FACTORS:
    print(f'\n=== Scale x{scale} ({SIZE_LABELS.get(scale, str(scale))}) ===')

    # Load fresh base datasets
    base_train_rdd, base_test_rdd = load_rdd(spark, filepath=DATA_PATH)
    base_train_df,  base_test_df  = load_dataframe(spark, filepath=DATA_PATH)

    # Repeat the training data to simulate a larger dataset.
    # We repeat only training — test stays at base size so accuracy is comparable.
    train_rdd = base_train_rdd
    train_df  = base_train_df
    for _ in range(scale - 1):
        train_rdd = sc.union([train_rdd, base_train_rdd])
        train_df  = train_df.union(base_train_df)

    train_rdd.cache()
    train_df.cache()
    n_train = train_rdd.count()
    print(f'  Training rows: {n_train}')

    for impl_name, train_fn, predict_fn, use_rdd in [
        ('RDD Baseline',  train_rdd_baseline,  predict_rdd_baseline,  True),
        ('RDD Optimised', train_rdd_optimized, predict_rdd_optimized, True),
        ('DF Baseline',   train_df_baseline,   predict_df_baseline,   False),
        ('DF Optimised',  train_df_optimized,  predict_df_optimized,  False),
    ]:
        train_data = train_rdd if use_rdd else train_df
        test_data  = base_test_rdd if use_rdd else base_test_df

        # --- Train ---
        t0 = time.time()
        raw_counts = train_fn(train_data)
        train_time = time.time() - t0

        # Build the shared probability table from raw counts
        if use_rdd:
            class_counts   = {k[6:]: v for k, v in raw_counts.items() if k.startswith('class_')}
            feature_counts = {k: v       for k, v in raw_counts.items() if k.startswith('feat_')}
        else:
            class_counts, feature_counts = raw_counts
        log_prob_table = compute_log_probs(class_counts, feature_counts, class_counts)

        # --- Predict ---
        t0 = time.time()
        pairs = predict_fn(test_data, log_prob_table)
        pred_time = time.time() - t0

        # --- Accuracy ---
        true_labels = [p[0] for p in pairs]
        pred_labels = [p[1] for p in pairs]
        acc = evaluate(pred_labels, true_labels)

        results.append({
            'impl':        impl_name,
            'scale':       scale,
            'size_label':  SIZE_LABELS.get(scale, str(scale)),
            'n_train':     n_train,
            'train_time':  train_time,
            'pred_time':   pred_time,
            'total_time':  train_time + pred_time,
            'accuracy':    acc,
        })
        print(f'  [{impl_name:14s}] train={train_time:.3f}s  pred={pred_time:.3f}s  acc={acc:.3f}')

    train_rdd.unpersist()
    train_df.unpersist()

print('\nAll experiments complete.')

In [ ]:
# Cell 5 — Print formatted results table

size_labels = [SIZE_LABELS.get(s, str(s)) for s in SCALE_FACTORS]
impl_names  = ['RDD Baseline', 'RDD Optimised', 'DF Baseline', 'DF Optimised']

# Build a lookup: (impl, size_label) -> total_time
lookup = {(r['impl'], r['size_label']): r['total_time'] for r in results}

# Header
col_w = 10
line  = '+' + '-' * 22 + '+' + ('+' + '-' * col_w) * len(size_labels) + '+'
print(line)
header = f"| {'Implementation':<20} |" + ''.join(f" {lbl:>{col_w - 1}} |" for lbl in size_labels)
print(header)
print(line)
for impl in impl_names:
    row = f'| {impl:<20} |'
    for lbl in size_labels:
        t = lookup.get((impl, lbl), float('nan'))
        row += f" {t:>{col_w - 4}.3f}s  |"
    print(row)
print(line)

# Speedup ratios
print()
for lbl in size_labels:
    rdd_base = lookup.get(('RDD Baseline',  lbl), None)
    rdd_opt  = lookup.get(('RDD Optimised', lbl), None)
    df_base  = lookup.get(('DF Baseline',   lbl), None)
    df_opt   = lookup.get(('DF Optimised',  lbl), None)

    if all(x is not None for x in [rdd_base, rdd_opt, df_base, df_opt]):
        best_rdd = min(rdd_base, rdd_opt)
        best_df  = min(df_base,  df_opt)
        print(f"{lbl}: RDD opt vs baseline = {rdd_base/rdd_opt:.2f}x faster")
        print(f"{lbl}: DF  opt vs baseline = {df_base/df_opt:.2f}x faster")
        print(f"{lbl}: Best RDD vs Best DF = {max(best_rdd, best_df)/min(best_rdd, best_df):.2f}x  "
              f"({'RDD' if best_rdd < best_df else 'DF'} wins)")
        print()

In [ ]:
# Cell 6 — Accuracy summary
#
# All four implementations should produce the same accuracy (within floating
# point rounding) because they all call the same compute_log_probs() and
# predict() functions from core/naive_bayes.py.
# If accuracies diverge, there is likely a bug in the count collection logic.

print('Accuracy by implementation (should be identical for all):')
for r in results:
    print(f"  {r['impl']:14s} x{r['scale']}: {r['accuracy']:.4f}")

# TODO: Expected accuracy on the full car evaluation dataset is ~87.39%
# (Zheng 2014). With the 5-row dummy dataset accuracy is not meaningful.
# Replace DATA_PATH in Cell 2 with the real file path before interpreting results.

spark.stop()